In [106]:
import pandas as pd
import numpy as np

In [107]:
df = pd.read_csv('ambitionbox.csv')

In [108]:
df.head(5)

,Unnamed: 0,name,rating,reviews,type,hq,old,employees
0,0,TCS,3.9,16.1k Reviews,Public,Mumbai + 156 more,52 years old,10000+ employees
1,1,Accenture,4.0,14.1k Reviews,Private,Dublin + 87 more,31 years old,10000+ employees
2,2,ICICI Bank,4.1,12.7k Reviews,Public,Mumbai + 724 more,26 years old,10000+ employees
3,3,Cognizant,3.9,12.1k Reviews,Private,Teaneck + 44 more,26 years old,10000+ employees
4,4,HDFC Bank,4.0,10.8k Reviews,Public,Mumbai + 689 more,26 years old,10000+ employees


In [109]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  9960 non-null   int64  
 1   name        9960 non-null   object 
 2   rating      9960 non-null   float64
 3   reviews     9960 non-null   object 
 4   type        9960 non-null   object 
 5   hq          8688 non-null   object 
 6   old         7909 non-null   object 
 7   employees   5814 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 622.6+ KB


In [110]:
df.describe()

,Unnamed: 0,rating
count,9960.000000,9960.000000
mean,4979.500000,3.857369
std,2875.348675,0.417702
min,0.000000,1.200000
25%,2489.750000,3.600000
50%,4979.500000,3.900000
75%,7469.250000,4.200000
max,9959.000000,5.000000


In [111]:
def rapihin_kolom_acak(row):
    # Ambil nilai baris dari 4 kolom yang bermasalah, jadikan string murni
    data_mentah = [str(row['type']), str(row['hq']), str(row['old']), str(row['employees'])]
    
    # Siapkan wadah kosong (NaN) untuk posisi yang benar
    tipe, lokasi, umur, karyawan = np.nan, np.nan, np.nan, np.nan
    
    # Cek satu-satu isi datanya dan masukkan ke kolom yang tepat berdasarkan kata kunci
    for teks in data_mentah:
        # Lewati kalau datanya memang kosong dari sananya
        if teks == 'nan' or teks == 'None' or teks.strip() == '' or pd.isna(teks):
            continue
            
        teks_lower = teks.lower()
        
        # Deteksi umur
        if 'years' in teks_lower or 'old' in teks_lower:
            umur = teks
        # Deteksi karyawan
        elif 'employees' in teks_lower:
            karyawan = teks
        # Deteksi jenis perusahaan (bisa ditambah kata kunci lain jika ada)
        elif 'public' in teks_lower or 'private' in teks_lower or 'other' in teks_lower:
            tipe = teks
        # Kalau tidak ada kata kunci di atas, asumsi kuat itu adalah nama kota / lokasi (HQ)
        else:
            lokasi = teks
            
    # Kembalikan posisinya ke urutan yang benar
    return pd.Series([tipe, lokasi, umur, karyawan])

# Hajar 4 kolom lo pakai fungsi ini (Pastikan urutan nama kolom di kiri sama dengan struktur df lo)
df[['type', 'hq', 'old', 'employees']] = df.apply(rapihin_kolom_acak, axis=1)

In [112]:
shifted_data = df[df['hq'].str.contains('years', na=False, case=False)]
shifted_data.head()

,Unnamed: 0,name,rating,reviews,type,hq,old,employees


In [113]:
shifted_data = df[df['old'].str.contains('employees', na=False, case=False)]
shifted_data.head()

,Unnamed: 0,name,rating,reviews,type,hq,old,employees


In [114]:
shifted_data = df[df['old'].str.contains('more', na=False, case=False)]
shifted_data.head()

,Unnamed: 0,name,rating,reviews,type,hq,old,employees
8882,8882,Lakshya Institute,3.5,30 Reviews,Private,NaN,Old Bridge + 9 more,1-50 employees


In [115]:
shifted_data = df[df['type'].str.contains('more', na=False, case=False)]
shifted_data.head(20)

,Unnamed: 0,name,rating,reviews,type,hq,old,employees
2513,2513,Seoyon E-hwa Automotive,4.1,103 Reviews,Slovak Republic + 5 more,NaN,18 years old,501-1000 employees


In [116]:
shifted_data = df[df['employees'].str.contains('years', na=False, case=False)]
shifted_data.head(20)

,Unnamed: 0,name,rating,reviews,type,hq,old,employees


In [117]:
cols_to_shift = ['type', 'hq']
kondisi = df['type'].str.contains('more', na=False, case=False)
df.loc[kondisi, cols_to_shift] = df.loc[kondisi, cols_to_shift].shift(1, axis=1)

In [118]:
cols_to_shift = ['hq', 'old']
kondisi = df['old'].str.contains('more', na=False, case=False)
df.loc[kondisi, cols_to_shift] = df.loc[kondisi, cols_to_shift].shift(-1, axis=1)

In [119]:
df['name']= df['name'].str.replace('...', '',regex=False).str.strip()
df['type'] = df['type'].replace('Central Public Sector Enterprises (CPSE)','Public')
df['hq']= df['hq'].str.replace('more', '',regex=True).str.strip()
df['old']= df['old'].str.replace('years old', '',regex=True).str.strip()
df['employees']= df['employees'].str.replace('+ employees', '',regex=False).str.strip()
df['employees']= df['employees'].str.replace('employees', '',regex=False).str.strip()

In [120]:
def perbaiki_reviews(teks):
    # Kalau datanya kosong (NaN), biarkan kosong
    if pd.isna(teks):
        return pd.NA
        
    # Ubah jadi string kecil semua dan buang kata "reviews" & spasi
    teks = str(teks).lower().replace('reviews', '').strip()
    
    try:
        # KONDISI 1: Kalau ada huruf 'k' (Ribuan)
        if 'k' in teks:
            angka = float(teks.replace('k', '')) # '1.1k' jadi 1.1
            return int(angka * 1000)             # 1.1 * 1000 jadi 1100
        
        # KONDISI 3: Angka biasa (Misal: 900)
        else:
            teks = teks.replace(',', '') # Jaga-jaga kalau ada koma misal "1,200"
            return int(float(teks))      # 900 tetap jadi 900
            
    except:
        # Kalau ada format aneh lain yang gagal di-convert, balikin kosong biar aman
        return pd.NA

# Terapkan fungsinya ke kolom reviews dan paksa tipe datanya jadi Int64
df['reviews'] = df['reviews'].apply(perbaiki_reviews).astype('Int64')

In [121]:
df[['hq_main', 'hq_branches']] = df['hq'].str.split(r'+', expand=True)

In [122]:
df.drop(columns=['hq'], inplace=True)

In [123]:
df.head()

,Unnamed: 0,name,rating,reviews,type,old,employees,hq_main,hq_branches
0,0,TCS,3.9,16100,Public,52,10000,Mumbai,156
1,1,Accenture,4.0,14100,Private,31,10000,Dublin,87
2,2,ICICI Bank,4.1,12700,Public,26,10000,Mumbai,724
3,3,Cognizant,3.9,12100,Private,26,10000,Teaneck,44
4,4,HDFC Bank,4.0,10800,Public,26,10000,Mumbai,689


In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   9960 non-null   int64  
 1   name         9960 non-null   object 
 2   rating       9960 non-null   float64
 3   reviews      9960 non-null   Int64  
 4   type         7277 non-null   object 
 5   old          7450 non-null   object 
 6   employees    7596 non-null   object 
 7   hq_main      9958 non-null   object 
 8   hq_branches  8933 non-null   object 
dtypes: Int64(1), float64(1), int64(1), object(6)
memory usage: 710.2+ KB


In [125]:
# 1. Bikin "Radar" mana baris yang ada tanda '-' (menghasilkan True/False)
mask_rentang = df['employees'].str.contains('-', na=False)

# 2. Hajar khusus baris yang True (ada '-')
parts = df.loc[mask_rentang, 'employees'].str.split('-', expand=True)
min_values = parts[0].astype(float)
max_values = parts[1].astype(float)

# 3. Timpa nilai di baris yang ada '-' dengan nilai tengahnya
df.loc[mask_rentang, 'employees'] = (min_values + max_values) / 2

# 4. Terakhir, ubah seluruh kolom jadi integer (Int64 agar aman dari NaN)
df['employees'] = df['employees'].astype(float).round().astype('Int64')

In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   9960 non-null   int64  
 1   name         9960 non-null   object 
 2   rating       9960 non-null   float64
 3   reviews      9960 non-null   Int64  
 4   type         7277 non-null   object 
 5   old          7450 non-null   object 
 6   employees    7596 non-null   Int64  
 7   hq_main      9958 non-null   object 
 8   hq_branches  8933 non-null   object 
dtypes: Int64(2), float64(1), int64(1), object(5)
memory usage: 719.9+ KB


In [127]:
kolom_numerik = ['old', 'hq_branches']
df[kolom_numerik] = df[kolom_numerik].astype(float).astype('Int64')

In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   9960 non-null   int64  
 1   name         9960 non-null   object 
 2   rating       9960 non-null   float64
 3   reviews      9960 non-null   Int64  
 4   type         7277 non-null   object 
 5   old          7450 non-null   Int64  
 6   employees    7596 non-null   Int64  
 7   hq_main      9958 non-null   object 
 8   hq_branches  8933 non-null   Int64  
dtypes: Int64(4), float64(1), int64(1), object(3)
memory usage: 739.3+ KB


In [129]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         9960 non-null   object 
 1   rating       9960 non-null   float64
 2   reviews      9960 non-null   Int64  
 3   type         7277 non-null   object 
 4   old          7450 non-null   Int64  
 5   employees    7596 non-null   Int64  
 6   hq_main      9958 non-null   object 
 7   hq_branches  8933 non-null   Int64  
dtypes: Int64(4), float64(1), object(3)
memory usage: 661.5+ KB


In [131]:
df['hq_branches'] = df['hq_branches'].fillna(0)
df['type'] = df['type'].fillna('Unknown')
df['hq_main'] = df['hq_main'].fillna('Unknown')

In [132]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9960 entries, 0 to 9959
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         9960 non-null   object 
 1   rating       9960 non-null   float64
 2   reviews      9960 non-null   Int64  
 3   type         9960 non-null   object 
 4   old          7450 non-null   Int64  
 5   employees    7596 non-null   Int64  
 6   hq_main      9960 non-null   object 
 7   hq_branches  9960 non-null   Int64  
dtypes: Int64(4), float64(1), object(3)
memory usage: 661.5+ KB


In [133]:
print("Duplikat: ", df.duplicated().sum())

Duplikat:  1


In [134]:
df.drop_duplicates(inplace=True)
print("Duplikat: ", df.duplicated().sum())

Duplikat:  0


In [135]:
# Mencari baris yang teksnya berubah kalau spasi di ujungnya dibuang
spasi_name = df[df['name'].notna() & (df['name'] != df['name'].astype(str).str.strip())]
spasi_type = df[df['type'].notna() & (df['type'] != df['type'].astype(str).str.strip())]
spasi_hq_main = df[df['hq_main'].notna() & (df['hq_main'] != df['hq_main'].astype(str).str.strip())]

print(f"Jumlah baris bermasalah (ada spasi gaib) di kolom 'name'   : {len(spasi_name)}")
print(f"Jumlah baris bermasalah (ada spasi gaib) di kolom 'type': {len(spasi_type)}")
print(f"Jumlah baris bermasalah (ada spasi gaib) di kolom 'hq_main': {len(spasi_hq_main)}")


Jumlah baris bermasalah (ada spasi gaib) di kolom 'name'   : 0
Jumlah baris bermasalah (ada spasi gaib) di kolom 'type': 0
Jumlah baris bermasalah (ada spasi gaib) di kolom 'hq_main': 8932


In [136]:
# Hitung jumlah variasi unik asli vs variasi setelah di-lowercase
variasi_asli = df['type'].dropna().nunique()
variasi_lower = df['type'].astype(str).str.lower().dropna().nunique()

print(f"Jumlah variasi unik asli  : {variasi_asli}")
print(f"Jumlah variasi unik (lower): {variasi_lower}")

if variasi_asli != variasi_lower:
    print("👉 FIX! Ada masalah kapital besar-kecil di kolom 'type'.")
else:
    print("👉 Aman! Tidak ada masalah kapital besar-kecil di kolom 'type'.")

Jumlah variasi unik asli  : 4
Jumlah variasi unik (lower): 4
👉 Aman! Tidak ada masalah kapital besar-kecil di kolom 'type'.


In [137]:
# Bersihkan spasi di awal dan akhir sekaligus untuk kolom hq_main
df['hq_main'] = df['hq_main'].astype(str).str.strip()

In [138]:
# Cek ulang apakah masih ada baris yang tidak sinkron setelah di-strip
spasi_hq_main_after = df[df['hq_main'].notna() & (df['hq_main'] != df['hq_main'].astype(str).str.strip())]
print("Jumlah baris bermasalah setelah dibersihkan:", len(spasi_hq_main_after))

Jumlah baris bermasalah setelah dibersihkan: 0


In [139]:
df.reset_index(drop=True, inplace=True)

In [140]:
# Simpan ke CSV baru, pastikan pake index=False biar kolom Unnamed: 0 gak lahir lagi!
df.to_csv('ambitionbox_cleaned_v4.csv', index=False, sep=';')